<a href="https://colab.research.google.com/github/YujiSue/Lecture/blob/master/%E3%81%98%E3%82%83%E3%82%93%E3%81%91%E3%82%93%E3%83%8E%E3%83%BC%E3%83%88.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# このノートについて
### GoogleのMediaPipeを利用した、人間の『手』の認識AIを利用するジャンケンゲームです
### カメラつきの機器で実行する必要があります

In [ ]:
#@title 1. はじめに
#@markdown ### まずは、このコードセルを実行してください
import cv2

!pip install -q mediapipe
!wget "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task" &> /dev/null

import mediapipe as mp
mp_hands = mp.tasks.vision.HandLandmarksConnections
mp_drawing = mp.tasks.vision.drawing_utils
mp_drawing_styles = mp.tasks.vision.drawing_styles

!wget 'https://drive.google.com/uc?export=download&id=13Fsf1U11Rep18RldDrVaQ7hpswDU2HLM' -O gu.png &> /dev/null
!wget 'https://drive.google.com/uc?export=download&id=1NWmkrRxy8pI8IxvgttwT37TvLVPE5ocx' -O pa.png &> /dev/null
!wget 'https://drive.google.com/uc?export=download&id=1KCVk7CKEdc-hm_OXHtVYxbBNUhfaYPta' -O choki.png &> /dev/null
#
rps = [cv2.imread('pa.png'),cv2.imread('choki.png'),cv2.imread('gu.png')]
captured = 'photo.jpg'
#
print('OK')

In [ ]:
#@title 2. 必要な関数の定義
#@markdown ### 上でOKが出たら、このコードセルを実行してください
import random
import numpy as np
import math
import base64
from IPython.display import Image, HTML, display, Javascript, clear_output
from google.colab.output import eval_js
from google.colab.patches import cv2_imshow

from mediapipe.tasks import python
from mediapipe.tasks.python import vision
base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')
options = vision.HandLandmarkerOptions(base_options=base_options, num_hands=1, min_hand_detection_confidence=0.1)
detector = vision.HandLandmarker.create_from_options(options)

# カメラで手を撮影
def useCam(file, quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const msg = document.createElement('h1');
      msg.id = 'message';
      msg.innerText = 'スタートボタンを押してね';
      div.appendChild(msg);

      const capture = document.createElement('button');
      capture.textContent = 'スタート';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      // 1. スタートボタンが押されるのを待つ
      await new Promise((resolve) => capture.onclick = resolve);
      capture.remove();

      // 2. 1秒ごとにメッセージを切り替える
      const sleep = ms => new Promise(resolve => setTimeout(resolve, ms));

      await sleep(1000);
      msg.innerText = 'じゃん';

      await sleep(1000);
      msg.innerText = 'けん';

      await sleep(1000);
      msg.innerText = 'ポン！';

      // カメラのプレビューを静止画像データとして取得
      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
  display(js)
  # 取り込んだデータを一時保存
  binary = eval_js('takePhoto({})'.format(quality))
  img = base64.b64decode(binary.split(',')[1])
  with open(file, 'wb') as f:
    f.write(img)

#
def randomHand():
  return random.randint(0, 2)

# ユークリッド距離の算出
def eucDist(pt1, pt2):
  return math.sqrt(math.pow(pt1.x - pt2.x, 2) + math.pow(pt1.y - pt2.y, 2))

# 距離データの取得
def dists(pts, ori):
  dist = []
  for pt in pts:
    dist.append(eucDist(pt, ori))
  return dist

# 判定
def discriminate(file, refs):
  ## リセット
  clear_output()
  ##
  try:
    ### 画像の読み込み
    img = mp.Image.create_from_file(file)
    ### 検出
    detected = detector.detect(img)
    hand_landmarks_list = detected.hand_landmarks

    ### 手が認識されたかチェック
    if len(hand_landmarks_list) == 0:
      return -1
    hand_landmarks = hand_landmarks_list[0]

    ### 認識されたポイント・ラインの描画
    annotated = np.copy(img.numpy_view())
    annotated = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    mp_drawing.draw_landmarks(
      annotated,
      hand_landmarks,
      mp_hands.HAND_CONNECTIONS,
      mp_drawing_styles.get_default_hand_landmarks_style(),
      mp_drawing_styles.get_default_hand_connections_style())
    cv2_imshow(annotated)

    ### 判別式
    wrist = hand_landmarks[mp_hands.HAND_CONNECTIONS[0].start]
    mcps = [
        hand_landmarks[mp_hands.HAND_THUMB_CONNECTIONS[0].start],
        hand_landmarks[mp_hands.HAND_INDEX_FINGER_CONNECTIONS[0].start],
        hand_landmarks[mp_hands.HAND_MIDDLE_FINGER_CONNECTIONS[0].start],
        hand_landmarks[mp_hands.HAND_RING_FINGER_CONNECTIONS[0].start],
        hand_landmarks[mp_hands.HAND_PINKY_FINGER_CONNECTIONS[0].start]
    ]
    tips = [
        hand_landmarks[mp_hands.HAND_THUMB_CONNECTIONS[2].end],
        hand_landmarks[mp_hands.HAND_INDEX_FINGER_CONNECTIONS[2].end],
        hand_landmarks[mp_hands.HAND_MIDDLE_FINGER_CONNECTIONS[2].end],
        hand_landmarks[mp_hands.HAND_RING_FINGER_CONNECTIONS[2].end],
        hand_landmarks[mp_hands.HAND_PINKY_FINGER_CONNECTIONS[2].end]
    ]
    dist1 = dists(mcps, wrist)
    dist2 = dists(tips, wrist)
    ave_dist = np.average(np.array(dist1))

    threshold = 1.25
    score = 0
    if threshold * ave_dist < dist2[0]:
      score += 1
    if threshold * ave_dist < dist2[1]:
      score += 3
    if threshold * ave_dist < dist2[2]:
      score += 3
    if threshold * ave_dist < dist2[3]:
      score += 1
    if threshold * ave_dist < dist2[4]:
      score += 1

    if 8 <= score:
      return 0
    elif 6 <= score:
      return 1
    elif score <= 1:
      return 2

    return -1

  except Exception as e:
    print(e)
    return -1

print('OK')

In [ ]:
#@title 3. 実行
#@markdown ### 最後にこのコードセルを実行してください<br>
#@markdown ### カメラの使用についての許可を求められるので許可してください<br>セキュリティが気になる場合は「今回のみ許可」のような限定許可を選んでください<br/>
#@markdown ###「じゃん」⇒「けん」の表示中、カメラになるべく手だけが写るようにしてください</font>

# カメラから手の画像を取得
useCam(captured)
# コンピュータの手が決定
compHand = randomHand()
# ユーザの手を判定
yourHand = discriminate(captured, rps)

# 結果表示
print('コンピュータの手は')
cv2_imshow(cv2.resize(rps[compHand], dsize=(200,200)))
print('あなたの出した手は...')
if yourHand == -1:
  print('判別できませんでした')
else:
  cv2_imshow(cv2.resize(rps[yourHand], dsize=(200,200)))
  if ((compHand + 1) % 3) == (yourHand % 3):
    print('あなたの勝利!!!')
  elif ((yourHand + 1) % 3) == (compHand % 3):
    print('残念！あなたの負け!')
  else:
    print('引き分け!')

##### Copyright 2023 The MediaPipe Authors. All Rights Reserved.
##### Modified by Yuji Suehiro 2026 for educational use.

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.